# **TASK 01**

In [58]:
#Mounting to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [59]:
#Move into the project folder and create data/
import os
os.chdir('/content/drive/MyDrive/CRM_Coursework')
os.makedirs('data', exist_ok=True)
print("Working in:", os.getcwd())

Working in: /content/drive/MyDrive/CRM_Coursework


In [60]:
#Install packages, set study parameters, print versions
!pip install yfinance lxml pyarrow -q

import io, json, os, random
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from datetime import datetime

START = "2021-01-01"
END   = "2025-01-01"
BENCH = "^GSPC"
BASELINE_DAYS = 20
WINDOW_DAYS   = 60
SPIKE_MULT    = 2

print("pandas", pd.__version__, "| numpy", np.__version__, "| yfinance", yf.__version__)

pandas 2.2.3 | numpy 2.1.3 | yfinance 0.2.66


## Task 1.1

In [61]:
#[Task 1.1] Extract S&P 500 ticker list and record retrieval date
URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
table = pd.read_html(io.StringIO(html))[0]
table["Symbol"] = table["Symbol"].str.replace(".", "-", regex=False)
table["retrieved_on"] = datetime.now().strftime("%Y-%m-%d %H:%M")
table.to_csv("data/sp500_constituents.csv", index=False)

retrieval_date = table["retrieved_on"].iloc[0]
tickers = table["Symbol"].tolist()
print("Retrieved:", retrieval_date, "|", len(tickers), "tickers")
table.head()

Retrieved: 2026-11-01 18:17 | 503 tickers


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded,retrieved_on
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902,2026-11-01 18:17
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916,2026-11-01 18:17
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888,2026-11-01 18:17
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888),2026-11-01 18:17
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989,2026-11-01 18:17


In [62]:
#[Task 1.1] Download prices, volume and ^GSPC with yfinance; save snapshot
CACHE = {"adj": "data/adj_close.parquet",
         "close": "data/close.parquet",
         "vol": "data/volume.parquet"}
all_tickers = tickers + [BENCH]

if all(os.path.exists(p) for p in CACHE.values()):
    adj, close, volume = (pd.read_parquet(CACHE[k]) for k in ("adj", "close", "vol"))
    source = "cached snapshot (parquet)"
else:
    frames = []
    for i in range(0, len(all_tickers), 100):          # 100 tickers at a time
        print("Downloading tickers", i, "to", min(i + 100, len(all_tickers)))
        frames.append(yf.download(all_tickers[i:i+100], start=START, end=END,
                                  auto_adjust=False, progress=False))
    raw = pd.concat(frames, axis=1)
    adj    = raw["Adj Close"].reindex(columns=all_tickers)
    close  = raw["Close"].reindex(columns=all_tickers)
    volume = raw["Volume"].reindex(columns=all_tickers)
    adj.to_parquet(CACHE["adj"])
    close.to_parquet(CACHE["close"])
    volume.to_parquet(CACHE["vol"])
    source = "live yfinance download"

print()
print("Source:", source)
print("Shape (days x tickers):", adj.shape)
print("First date:", adj.index.min().date(), "| Last date:", adj.index.max().date())
assert adj.index.max() < pd.Timestamp(END)


Source: cached snapshot (parquet)
Shape (days x tickers): (1005, 504)
First date: 2021-01-04 | Last date: 2024-12-31


In [63]:
# [Task 1.1] Check that the snapshot files were saved
for name in sorted(os.listdir("data")):
    print(f"{name}  ({os.path.getsize('data/' + name) / 1e6:.2f} MB)")

adj_close.parquet  (3.67 MB)
big_moves_check.csv  (0.00 MB)
close.parquet  (3.47 MB)
common_share_set.csv  (0.04 MB)
exclusion_log.csv  (0.00 MB)
feature_table.csv  (0.04 MB)
inclusion_exclusion_log.csv  (0.01 MB)
run_metadata.json  (0.00 MB)
sp500_constituents.csv  (0.06 MB)
volume.parquet  (3.38 MB)
weekly_returns.csv  (2.12 MB)


In [64]:
# [Task 1.1] Preview missing data per ticker
print("Missing values per ticker (top 10 worst):")
print(adj.isna().sum().sort_values(ascending=False).head(10))
print()
adj.iloc[:3, :5]

Missing values per ticker (top 10 worst):
Ticker
HONA    1005
FDXF    1005
Q       1005
VYLR    1005
SNDK    1005
GEV      812
SOLV     811
RDDT     808
VLTO     692
KVUE     587
dtype: int64



Ticker,MMM,AOS,ABT,ABBV,ACN
Date,,,,,
2021-01-04,117.860291,48.167667,97.269699,84.219162,232.029312
2021-01-05,117.661430,48.480339,98.473213,85.090004,233.350250
2021-01-06,119.451241,50.150841,98.268166,84.354958,235.901642


## Task 1.2

In [65]:
# [Task 1.2] Validation checks and inclusion/exclusion log
# --- global checks ---
print("Duplicate dates:", adj.index.duplicated().sum())
print("Dates sorted oldest to newest:", adj.index.is_monotonic_increasing)

cal = adj[BENCH].dropna().index          # the trading calendar from the benchmark
print("Benchmark trading days:", len(cal))
print("Benchmark missing values:", adj[BENCH].isna().sum())

# --- check every share one by one ---
rows = []
for t in tickers:
    a = adj[t].reindex(cal)
    c = close[t].reindex(cal)
    v = volume[t].reindex(cal)
    reasons = []

    if a.isna().all():
        reasons.append("no data returned")
    else:
        first = a.first_valid_index()
        if first > cal[0]:
            reasons.append(f"late start ({first.date()})")
        gaps = int(a.loc[first:].isna().sum()
                   + c.loc[first:].isna().sum()
                   + v.loc[first:].isna().sum())
        if gaps:
            reasons.append(f"{gaps} missing values after start")
        if (a <= 0).any() or (c <= 0).any():
            reasons.append("non-positive price")
        if (v < 0).any():
            reasons.append("negative volume")
        if (v == 0).any():
            reasons.append(f"{int((v == 0).sum())} zero-volume days")

    rows.append({"ticker": t, "included": len(reasons) == 0, "reason": "; ".join(reasons)})

status = pd.DataFrame(rows)
valid = status.loc[status["included"], "ticker"].tolist()
exclusion_log = status.loc[~status["included"]]
exclusion_log.to_csv("data/exclusion_log.csv", index=False)

print()
print("Total tickers checked:", len(status))
print("Included:", len(valid))
print("Excluded:", len(exclusion_log))
exclusion_log

Duplicate dates: 0
Dates sorted oldest to newest: True
Benchmark trading days: 1005
Benchmark missing values: 0

Total tickers checked: 503
Included: 485
Excluded: 18


,ticker,included,reason
40,APP,False,late start (2021-04-15)
116,COIN,False,late start (2021-04-14)
123,CEG,False,late start (2022-01-19)
181,EXE,False,late start (2021-02-10)
192,FDXF,False,no data returned
193,FERG,False,27 zero-volume days
210,GEHC,False,late start (2022-12-15)
211,GEV,False,late start (2024-03-27)
233,HONA,False,no data returned
269,KVUE,False,late start (2023-05-04)


In [66]:
# [Task 1.2] Investigate the zero-volume exclusions
for t in ["FERG", "SW"]:
    v = volume[t]
    print(t, "| first date with data:", v.first_valid_index().date(),
          "| zero-volume days:", int((v == 0).sum()),
          "| first nonzero-volume date:", v[v > 0].index.min().date())
    print(pd.DataFrame({"close": close[t], "volume": v}).loc[v[v > 0].index.min() - pd.Timedelta(days=5):].head(8))
    print()

FERG | first date with data: 2021-01-04 | zero-volume days: 27 | first nonzero-volume date: 2021-01-04
                 close  volume
Date                          
2021-01-04  120.279999     400
2021-01-05  118.599998     300
2021-01-06  121.300003     100
2021-01-07  121.300003       0
2021-01-08  121.300003       0
2021-01-11  124.199997    1000
2021-01-12  124.199997       0
2021-01-13  124.199997       0

SW | first date with data: 2021-01-04 | zero-volume days: 424 | first nonzero-volume date: 2021-01-04
                close  volume
Date                         
2021-01-04  47.610001     500
2021-01-05  47.610001       0
2021-01-06  47.610001       0
2021-01-07  47.610001     500
2021-01-08  49.209999   12500
2021-01-11  47.900002    1600
2021-01-12  47.900002       0
2021-01-13  47.900002       0



In [67]:
# [Task 1.2] Corporate actions 1: large one-day moves
# ---- Check 1: huge one-day moves in Close (possible unadjusted splits) ----
daily = close[valid].pct_change(fill_method=None)
big = daily.stack()
big = big[big.abs() > 0.4]
big_moves = big.reset_index()
big_moves.columns = ["date", "ticker", "daily_change"]
big_moves.to_csv("data/big_moves_check.csv", index=False)
print("Moves bigger than 40% in one day:", len(big_moves))
print(big_moves.sort_values("daily_change").head(10))

Moves bigger than 40% in one day: 8
        date ticker  daily_change
5 2024-04-11     GL     -0.531402
1 2022-12-07   CVNA     -0.429210
6 2024-07-26   DXCM     -0.406583
0 2022-08-05   CVNA      0.400716
4 2023-07-19   CVNA      0.402010
2 2023-01-12   CVNA      0.460000
3 2023-06-08   CVNA      0.560206
7 2024-11-15     BE      0.591867


In [68]:
# [Task 1.2] Corporate actions 2: Close vs Adj Close
# ---- Check 2: Close vs Adj Close (dividend adjustment) ----
ratio = adj[valid] / close[valid]
print("Adj/Close on the last date (should be 1):", ratio.iloc[-1].round(6).describe()[["min", "max"]].tolist())
print()
print("Cumulative dividend adjustment at the START of the period (1 - ratio):")
print((1 - ratio.iloc[0]).describe())

Adj/Close on the last date (should be 1): [0.87721, 1.0]

Cumulative dividend adjustment at the START of the period (1 - ratio):
count    485.000000
mean       0.096928
std        0.077809
min        0.000000
25%        0.027537
50%        0.090180
75%        0.153165
max        0.353954
Name: 2021-01-04 00:00:00, dtype: float64


In [69]:
# [Task 1.2] Corporate actions 3: test known split dates
# ---- Check 3: known stock splits ----
splits_to_test = {"NVDA": "2024-06-10", "AMZN": "2022-06-06", "TSLA": "2022-08-25",
                  "CMG": "2024-06-26", "AVGO": "2024-07-15"}
for t, d in splits_to_test.items():
    if t in valid:
        i = close.index.get_indexer([pd.Timestamp(d)], method="nearest")[0]
        price_chg = close[t].iloc[i] / close[t].iloc[i-1] - 1
        vol_x = volume[t].iloc[i] / volume[t].iloc[i-1]
        print(f"{t} {close.index[i].date()} | close change: {price_chg:+.1%} | volume x{vol_x:.2f}")

NVDA 2024-06-10 | close change: +0.7% | volume x0.76
AMZN 2022-06-06 | close change: +2.0% | volume x1.39
TSLA 2022-08-25 | close change: -0.3% | volume x0.93
CMG 2024-06-26 | close change: +0.3% | volume x1.15
AVGO 2024-07-15 | close change: +0.8% | volume x0.76


In [70]:
# [Task 1.2] Confirm splits for the large-move shares
for t in ["GL", "CVNA", "DXCM", "BE"]:
    s = yf.Ticker(t).splits
    s = s.loc["2021-01-01":"2024-12-31"] if len(s) else s
    print(t, "| splits in study period:", "none" if len(s) == 0 else s.to_dict())

GL | splits in study period: none
CVNA | splits in study period: none
DXCM | splits in study period: {Timestamp('2022-06-13 00:00:00-0400', tz='America/New_York'): 4.0}
BE | splits in study period: none


In [71]:
# [Task 1.2] Dividend adjustment inside the study period
ratio = adj[valid] / close[valid]
in_period = 1 - ratio.iloc[0] / ratio.iloc[-1]    # dividends paid between 2021 and end 2024
print(in_period.describe())
print("Largest:", in_period.sort_values(ascending=False).head(5).round(3).to_dict())

count    4.850000e+02
mean     6.842707e-02
std      5.656161e-02
min     -1.160065e-07
25%      1.655113e-02
50%      5.992601e-02
75%      1.098999e-01
max      2.741108e-01
dtype: float64
Largest: {'MO': 0.274, 'T': 0.254, 'KMI': 0.22, 'LYB': 0.219, 'OKE': 0.213}


In [72]:
# [Task 1.2] Friday-ending weekly prices; drop incomplete weeks; check alignment
weekly_all = adj[valid + [BENCH]].resample("W-FRI").last()
days_in_week = adj[BENCH].resample("W-FRI").count()

week_end = weekly_all.index
week_start = week_end - pd.Timedelta(days=4)                  # the Monday of that week
last_study_day = pd.Timestamp(END) - pd.Timedelta(days=1)     # 31 Dec 2024
keep = (week_start >= pd.Timestamp(START)) & (week_end <= last_study_day)

print("Dropped weeks:", [d.date() for d in week_end[~keep]])
weekly = weekly_all[keep]

print("Weekly prices shape:", weekly.shape)
print("First week:", weekly.index[0].date(), "| Last week:", weekly.index[-1].date())
print("Missing weekly prices:", int(weekly.isna().sum().sum()))
print("Every gap is exactly 7 days:", (weekly.index.to_series().diff().dropna() == pd.Timedelta(days=7)).all())
print("Trading days per retained week:")
print(days_in_week[weekly.index].value_counts().sort_index())

Dropped weeks: [datetime.date(2025, 1, 3)]
Weekly prices shape: (208, 486)
First week: 2021-01-08 | Last week: 2024-12-27
Missing weekly prices: 0
Every gap is exactly 7 days: True
Trading days per retained week:
^GSPC
4     37
5    171
Name: count, dtype: int64


## Task 1.3

In [73]:
# [Task 1.3.1 / 1.3.2] Weekly returns, mean return, weekly beta, independent checks
rets = weekly.pct_change(fill_method=None).iloc[1:]     # first week only gives the base price
n_obs = len(rets)
print("Aligned weekly return observations:", n_obs)
assert rets.isna().sum().sum() == 0

mean_ret = rets[valid].mean()                           # arithmetic mean

bench = rets[BENCH]
beta = rets[valid].apply(lambda s: s.cov(bench)) / bench.var()

# --- independent checks ---
t = valid[0]
wk, prev = weekly.index[10], weekly.index[9]
by_hand = adj[t].loc[:wk].iloc[-1] / adj[t].loc[:prev].iloc[-1] - 1
assert abs(by_hand - rets.loc[wk, t]) < 1e-12
print("Weekly return check OK for", t)

slope = np.polyfit(bench.values, rets[t].values, 1)[0]
assert abs(slope - beta[t]) < 1e-9
print("Beta check OK for", t, ":", round(beta[t], 3))
print("Benchmark self-beta:", round(bench.cov(bench) / bench.var(), 6))

print()
print(mean_ret.describe())
print(beta.describe())

Aligned weekly return observations: 207
Weekly return check OK for MMM
Beta check OK for MMM : 0.869
Benchmark self-beta: 1.0

count    485.000000
mean       0.002868
std        0.002644
min       -0.004468
25%        0.001183
50%        0.002623
75%        0.004208
max        0.017772
dtype: float64
count    485.000000
mean       0.971952
std        0.405941
min        0.055453
25%        0.666581
50%        0.948044
75%        1.179165
max        3.778560
dtype: float64


In [74]:
# [Task 1.3] Inspect extreme betas and mean returns
print("Highest betas:", beta.sort_values(ascending=False).head(5).round(2).to_dict())
print("Lowest betas:", beta.sort_values().head(5).round(2).to_dict())
print("Highest mean returns:", (mean_ret.sort_values(ascending=False).head(5) * 100).round(2).to_dict())

Highest betas: {'CVNA': 3.78, 'XYZ': 2.5, 'COHR': 2.14, 'NVDA': 2.13, 'TWLO': 2.07}
Lowest betas: {'GIS': 0.06, 'SJM': 0.18, 'MRK': 0.2, 'BMY': 0.23, 'KHC': 0.23}
Highest mean returns: {'SMCI': 1.78, 'NVDA': 1.36, 'VRT': 1.26, 'VST': 1.14, 'FIX': 1.13}


## Task 1.4

In [75]:
# [Task 1.4] Observation date, rolling 20-session threshold, high-volume flags
c = close[valid]
v = volume[valid]

obs_date = close[valid + [BENCH]].dropna().index.max()
print("Observation date:", obs_date.date())

baseline  = v.rolling(BASELINE_DAYS).mean().shift(1)    # mean of the PREVIOUS 20 sessions
threshold = SPIKE_MULT * baseline
flag = (v > threshold) & baseline.notna()               # strictly greater than

print("Total event flags:", int(flag.sum().sum()))
print("First possible event date:", flag.index[BASELINE_DAYS].date())

Observation date: 2024-12-31
Total event flags: 18222
First possible event date: 2021-02-02


In [76]:
# [Task 1.4.1 / 1.4.2 / 1.4.3] Compute Recency, Frequency, Monetary
# Recency: calendar days from the latest event to the observation date
last_event = flag.apply(lambda s: s.index[s].max())     # NaT when no event exists
R = (obs_date - last_event).dt.days                     # NaN when no event, never 0

# Frequency and Monetary: latest 60 trading sessions up to the observation date
sessions = v.loc[:obs_date].index
window = sessions[-WINDOW_DAYS:]
F = flag.loc[window].sum()
M = (c.loc[window] * v.loc[window]).sum()               # Close x Volume, summed, in USD

print("Window:", window[0].date(), "to", window[-1].date(), "| sessions:", len(window))
print("Shares with R = 0 (event on observation date):", int((R == 0).sum()))
print("Shares with no event (R missing):", int(R.isna().sum()))
print()
print(pd.DataFrame({"R": R, "F": F, "M": M}).describe())

Window: 2024-10-07 to 2024-12-31 | sessions: 60
Shares with R = 0 (event on observation date): 0
Shares with no event (R missing): 0

                R           F             M
count  485.000000  485.000000  4.850000e+02
mean    20.393814    2.985567  3.945567e+10
std     21.362369    1.329030  1.292672e+11
min      4.000000    0.000000  1.297483e+09
25%     11.000000    2.000000  1.017679e+10
50%     11.000000    3.000000  1.670905e+10
75%     14.000000    4.000000  3.236510e+10
max    222.000000    8.000000  1.847697e+12


## Task 1.5

In [77]:
# [Task 1.5] Verify RFM with an independent loop
def rfm_manual(t):
    vol = v[t].loc[:obs_date].tolist()
    px  = c[t].loc[:obs_date].tolist()
    dates = v[t].loc[:obs_date].index
    events = [dates[i] for i in range(BASELINE_DAYS, len(vol))
              if vol[i] > SPIKE_MULT * (sum(vol[i-BASELINE_DAYS:i]) / BASELINE_DAYS)]
    R_ = (obs_date - events[-1]).days if events else np.nan
    F_ = sum(1 for e in events if e >= dates[-WINDOW_DAYS])
    M_ = sum(px[i] * vol[i] for i in range(len(vol) - WINDOW_DAYS, len(vol)))
    return R_, F_, M_

random.seed(42)
for t in random.sample(valid, 10):
    r_, f_, m_ = rfm_manual(t)
    assert (np.isnan(r_) and np.isnan(R[t])) or r_ == R[t], t
    assert f_ == F[t], t
    assert abs(m_ - M[t]) < 1e-6 * abs(M[t]), t
print("Manual RFM matches the pandas calculation for 10 random shares")

Manual RFM matches the pandas calculation for 10 random shares


In [78]:
# [Task 1.5] History, threshold alignment and boundary checks
n_sessions = v.loc[:obs_date].notna().sum()
print("Minimum sessions per share:", n_sessions.min(), "| needed at least", BASELINE_DAYS + WINDOW_DAYS)
print("Baseline available for every day in the window:", baseline.loc[window].notna().all().all())

# threshold alignment test: the baseline must NOT include the tested day
t = valid[0]
i = 100
expected = v[t].iloc[i-BASELINE_DAYS:i].mean()
print("Baseline uses only previous days:", abs(baseline[t].iloc[i] - expected) < 1e-9)

# exact-boundary rule: volume equal to the threshold must NOT qualify
print("Any flagged day with volume == threshold:", bool(((v == threshold) & flag).any().any()))

Minimum sessions per share: 1005 | needed at least 80
Baseline available for every day in the window: True
Baseline uses only previous days: True
Any flagged day with volume == threshold: False


In [79]:
# [Task 1.5] Worked RFM example for one share
t = "AAPL"        # change if you prefer another share
ev = last_event[t]
print(f"{t}: latest event date  = {ev.date()}")
print(f"  volume that day      = {v[t][ev]:,.0f}")
print(f"  20-day baseline      = {baseline[t][ev]:,.0f}")
print(f"  threshold (2x)       = {threshold[t][ev]:,.0f}")
print(f"R = {obs_date.date()} minus {ev.date()} = {R[t]} calendar days")
print()
print("Flagged days inside the 60-session window:",
      [d.date() for d in window[flag.loc[window, t].values]])
print("F =", F[t])
print()
daily_value = c[t].loc[window] * v[t].loc[window]
print("First 3 daily traded values (Close x Volume):")
print(daily_value.head(3).round(0))
print(f"M = sum of 60 daily values = ${M[t]:,.0f}")

AAPL: latest event date  = 2024-12-20
  volume that day      = 147,495,300
  20-day baseline      = 45,004,605
  threshold (2x)       = 90,009,210
R = 2024-12-31 minus 2024-12-20 = 11 calendar days

Flagged days inside the 60-session window: [datetime.date(2024, 11, 25), datetime.date(2024, 12, 20)]
F = 2

First 3 daily traded values (Close x Volume):
Date
2024-10-07    8.757952e+09
2024-10-08    7.192062e+09
2024-10-09    7.710501e+09
Name: AAPL, dtype: float64
M = sum of 60 daily values = $623,303,377,707


In [80]:
# [Task 1.5] Build feature table, summaries, skewness; save outputs
features = pd.DataFrame({
    "mean_weekly_return": mean_ret,
    "weekly_beta": beta,
    "n_weeks": rets[valid].notna().sum(),
    "R": R, "F": F, "M": M,
})
features["sector"] = table.set_index("Symbol")["GICS Sector"].reindex(features.index)

core = ["mean_weekly_return", "weekly_beta", "R", "F", "M"]
common = features.dropna(subset=core)

print("Included shares:", len(features), "| common set with all 5 features:", len(common))
print("Shares excluded for missing R:", features.index[features["R"].isna()].tolist())
print()
print(features[core].describe().T)
print()
print("Skewness:")
print(features[core].skew().round(2))

features.to_csv("data/feature_table.csv")
common.to_csv("data/common_share_set.csv")
rets.to_csv("data/weekly_returns.csv")

meta = {"retrieved_on": retrieval_date, "data_source": source, "yfinance": yf.__version__,
        "pandas": pd.__version__, "numpy": np.__version__,
        "start": START, "end_exclusive": END, "observation_date": str(obs_date.date()),
        "weekly_return_obs": int(n_obs), "n_tickers_listed": len(tickers),
        "n_included": len(valid), "n_excluded": len(tickers) - len(valid),
        "n_common_set": int(len(common))}
json.dump(meta, open("data/run_metadata.json", "w"), indent=2)
print(meta)

Included shares: 485 | common set with all 5 features: 485
Shares excluded for missing R: []

                    count          mean           std           min  \
mean_weekly_return  485.0  2.867806e-03  2.643954e-03 -4.468093e-03   
weekly_beta         485.0  9.719521e-01  4.059407e-01  5.545297e-02   
R                   485.0  2.039381e+01  2.136237e+01  4.000000e+00   
F                   485.0  2.985567e+00  1.329030e+00  0.000000e+00   
M                   485.0  3.945567e+10  1.292672e+11  1.297483e+09   

                             25%           50%           75%           max  
mean_weekly_return  1.182856e-03  2.622774e-03  4.207596e-03  1.777248e-02  
weekly_beta         6.665813e-01  9.480437e-01  1.179165e+00  3.778560e+00  
R                   1.100000e+01  1.100000e+01  1.400000e+01  2.220000e+02  
F                   2.000000e+00  3.000000e+00  4.000000e+00  8.000000e+00  
M                   1.017679e+10  1.670905e+10  3.236510e+10  1.847697e+12  

Skewness:
mean_w

In [81]:
# R is bunched: which values are most common?
print(R.value_counts().head(8))
print("Latest event date counts:")
print(last_event.dt.date.value_counts().head(5))

11     353
36      17
55      11
60       6
46       6
25       5
102      5
26       5
Name: count, dtype: int64
Latest event date counts:
2024-12-20    353
2024-11-25     17
2024-11-06     11
2024-11-01      6
2024-11-15      6
Name: count, dtype: int64


In [82]:
# a full inclusion AND exclusion log (the brief asks for both)
status.to_csv("data/inclusion_exclusion_log.csv", index=False)
print(status["included"].value_counts())

included
True     485
False     18
Name: count, dtype: int64


In [83]:
# final file check: you should see about 11 files
for name in sorted(os.listdir("data")):
    print(f"{name}  ({os.path.getsize('data/' + name) / 1e6:.2f} MB)")

adj_close.parquet  (3.67 MB)
big_moves_check.csv  (0.00 MB)
close.parquet  (3.47 MB)
common_share_set.csv  (0.04 MB)
exclusion_log.csv  (0.00 MB)
feature_table.csv  (0.04 MB)
inclusion_exclusion_log.csv  (0.01 MB)
run_metadata.json  (0.00 MB)
sp500_constituents.csv  (0.06 MB)
volume.parquet  (3.38 MB)
weekly_returns.csv  (2.12 MB)
